# Exercise 1 — SMA and EMA

Moving averages smooth out price noise and reveal trends. The Simple Moving Average (SMA) gives equal weight to all prices in the window. The Exponential Moving Average (EMA) gives more weight to recent prices, making it more responsive to new information.

In [ ]:
import pandas as pd, math

def _synthetic(n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

# ── Exercise: implement sma and ema ──────────────────────────────────────────

def sma(series, window=20):
    """Simple Moving Average.

    Args:
        series : pd.Series of prices
        window : look-back period (default 20)

    Returns:
        pd.Series — first window-1 values are NaN
    """
    # TODO: return series.rolling(window=window).mean()
    return pd.Series([float("nan")] * len(series), index=series.index)


def ema(series, window=20):
    """Exponential Moving Average.

    Args:
        series : pd.Series of prices
        window : span parameter (default 20); alpha = 2 / (window + 1)

    Returns:
        pd.Series — no NaN (starts from the first value)
    """
    # TODO: return series.ewm(span=window, adjust=False).mean()
    return pd.Series([float("nan")] * len(series), index=series.index)


### Checks

In [ ]:
checks = 0

# 1 — sma: first window-1 values are NaN; window-th value is not NaN
try:
    close = _synthetic()["Close"]
    s = sma(close, 10)
    assert isinstance(s, pd.Series) and len(s) == len(close)
    assert s.iloc[:9].isna().all(), "first 9 values should be NaN"
    assert not pd.isna(s.iloc[9]), "index 9 should be the first non-NaN"
    checks += 1; print("✅ 1 sma: first window-1 NaN, then non-NaN")
except Exception as e:
    print("❌ 1:", e)

# 2 — sma: value at window-1 equals mean of first window elements
try:
    s = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0])
    result = sma(s, 3)
    assert abs(result.iloc[2] - 2.0) < 1e-9, f"expected 2.0, got {result.iloc[2]}"
    assert abs(result.iloc[3] - 3.0) < 1e-9, f"expected 3.0, got {result.iloc[3]}"
    assert abs(result.iloc[4] - 4.0) < 1e-9, f"expected 4.0, got {result.iloc[4]}"
    checks += 1; print("✅ 2 sma values are exact rolling means")
except Exception as e:
    print("❌ 2:", e)

# 3 — sma: constant series returns constant (where not NaN)
try:
    const = pd.Series([5.0] * 30)
    s = sma(const, 10)
    non_nan = s.dropna()
    assert len(non_nan) == 21 and (non_nan == 5.0).all()
    checks += 1; print("✅ 3 sma of constant series returns constant")
except Exception as e:
    print("❌ 3:", e)

# 4 — ema: no NaN values; same length as input
try:
    close = _synthetic()["Close"]
    e = ema(close, 20)
    assert isinstance(e, pd.Series) and len(e) == len(close)
    assert not e.isna().any(), "ema should have no NaN values"
    checks += 1; print("✅ 4 ema returns same length Series with no NaN")
except Exception as e:
    print("❌ 4:", e)

# 5 — ema: converges toward a new level after a step change
try:
    # Series: 10 zeros, then 10 tens — ema should end up close to 10
    s = pd.Series([0.0] * 10 + [10.0] * 20)
    e = ema(s, 3)
    assert e.iloc[-1] > 9.0, f"expected ema to converge to ~10, got {e.iloc[-1]:.4f}"
    checks += 1; print("✅ 5 ema converges to new price level after a step change")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
